In [3]:
import cv2
import mediapipe as mp
import csv
import time
import numpy as np

# Configuración de MediaPipe Hands
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils


In [5]:
# === CONFIGURACIÓN DEL USUARIO ===
LABEL = "A"        # 👈 Cambia la etiqueta según la seña
OUTPUT_FILE = "dataset.csv"
NUM_SAMPLES = 200      # 👈 Cuántos ejemplos por seña quieres

# Función: normalizar landmarks (traslación + escala)
def normalize_landmarks(hand_landmarks):
    coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark])

    # 1) Traslación: centramos en la muñeca (landmark 0)
    wrist = coords[0]
    coords -= wrist

    # 2) Escala: normalizamos por distancia muñeca → base dedo medio (landmark 9)
    scale = np.linalg.norm(coords[9])  # distancia al punto 9 después de traslación
    if scale > 0:
        coords /= scale

    return coords.flatten().tolist()

# === INICIALIZAR CSV ===
with open(OUTPUT_FILE, "a", newline="") as f:
    writer = csv.writer(f)
    if f.tell() == 0:  # Si el archivo está vacío, escribir cabecera
        header = [f"f{i}" for i in range(126)] + ["num_hands", "label"]
        writer.writerow(header)

# === CAPTURA DE CÁMARA ===
cap = cv2.VideoCapture(0)

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
) as hands:
    
    sample_count = 0
    print(f"Grabando señas para: {LABEL}")
    time.sleep(2)  # Pequeña pausa antes de comenzar

    while cap.isOpened() and sample_count < NUM_SAMPLES:
        success, frame = cap.read()
        if not success:
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(frame_rgb)

        frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

        row = []
        num_hands = 0
        if results.multi_hand_landmarks:
            num_hands = len(results.multi_hand_landmarks)
            for hand_landmarks in results.multi_hand_landmarks:
                row.extend(normalize_landmarks(hand_landmarks))

                # Dibujar landmarks en la imagen
                mp_drawing.draw_landmarks(frame_bgr, hand_landmarks, mp_hands.HAND_CONNECTIONS)

        # Rellenar con ceros si no hay 2 manos completas
        while len(row) < 126:
            row.append(0.0)

        if any(val != 0.0 for val in row):  # Si detectó al menos una mano
            row.append(num_hands)   # 👈 guardamos cantidad de manos
            row.append(LABEL)
            with open(OUTPUT_FILE, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(row)
            sample_count += 1

        # Mostrar progreso
        cv2.putText(frame_bgr, f"Label: {LABEL}  Sample: {sample_count}/{NUM_SAMPLES}",
                    (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        cv2.imshow("Recolector de Dataset", frame_bgr)

        if cv2.waitKey(1) & 0xFF == 27:  # ESC para salir
            break

    print(f"✅ Captura terminada para la seña {LABEL} ({sample_count} muestras)")

cap.release()
cv2.destroyAllWindows()

Grabando señas para: A
✅ Captura terminada para la seña A (200 muestras)
